In [66]:
import requests
import pandas as pd
import numpy as np
import datetime

fixture_file = '../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv'
fixture_df = pd.read_csv(fixture_file)

fixture_df['date'] = pd.to_datetime(fixture_df['Date']).dt.tz_localize(None)
fixture_df['home_team'] = fixture_df['HomeTeam'].str.strip().str.lower()
fixture_df['away_team'] = fixture_df['AwayTeam'].str.strip().str.lower()

now = pd.to_datetime(datetime.datetime.now()).tz_localize(None)
future_fixture_df = fixture_df[fixture_df['date'] >= now].copy()
future_fixture_df['date'] = future_fixture_df['date'].dt.normalize()
future_fixture_df = future_fixture_df.copy()

API_KEY = 'f91688b2469810e18dbf6649b7d462fe'
sport_key = 'soccer_epl'
region = 'uk,eu'
markets = 'h2h,totals,spreads'
url = (f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds/"
       f"?apiKey={API_KEY}&regions={region}&markets={markets}")
response = requests.get(url)
data = response.json()

if not isinstance(data, list):
    print("API error or quota issue:", data)
    raise Exception("Odds API error.")

all_rows, all_colnames = [], set()
for event in data:
    ev = {
        "date": pd.to_datetime(event.get("commence_time")).tz_localize(None).normalize(),
        "home_team": event.get("home_team").strip().lower(),
        "away_team": event.get("away_team").strip().lower()
    }
    for bookmaker in event.get("bookmakers", []):
        book = bookmaker.get("key", "")
        for market in bookmaker.get("markets", []):
            mkt = market.get("key", "")
            for out in market.get("outcomes", []):
                if mkt == "h2h":
                    if out["name"] == event["home_team"]:
                        col = f"{book}_H"
                    elif out["name"] == event["away_team"]:
                        col = f"{book}_A"
                    elif out["name"].lower() == "draw":
                        col = f"{book}_D"
                    else:
                        col = f"{book}_h2h_{out['name']}"
                    ev[col] = out["price"]
                    all_colnames.add(col)
                elif mkt == "totals":
                    sign = ">" if out["name"].lower() == "over" else "<"
                    pt = out.get("point", "")
                    col = f"{book}_{sign}{pt}"
                    ev[col] = out["price"]
                    all_colnames.add(col)
                elif mkt == "spreads":
                    side = "H" if out["name"] == event["home_team"] else "A"
                    pt = out.get("point", "")
                    col = f"{book}_AH{side}_{pt}"
                    ev[col] = out["price"]
                    all_colnames.add(col)
    all_rows.append(ev)
odds_df = pd.DataFrame(all_rows)
odds_df['date'] = pd.to_datetime(odds_df['date']).dt.tz_localize(None).dt.normalize()
odds_df['home_team'] = odds_df['home_team'].str.strip().str.lower()
odds_df['away_team'] = odds_df['away_team'].str.strip().str.lower()
odds_df = odds_df.copy()

merged = future_fixture_df.merge(
    odds_df,
    on=['date', 'home_team', 'away_team'],
    how='left',
    suffixes=('', '_odds')
)

# --- 4. Fill odds columns as best as possible ---
market_map = {
    'B365H': ['bet365_H', 'pinnacle_H', 'williamhill_H', 'betway_H'],
    'B365D': ['bet365_D', 'pinnacle_D', 'williamhill_D', 'betway_D'],
    'B365A': ['bet365_A', 'pinnacle_A', 'williamhill_A', 'betway_A'],
    'PSH':    ['pinnacle_H', 'bet365_H', 'williamhill_H'],
    'PSD':    ['pinnacle_D', 'bet365_D', 'williamhill_D'],
    'PSA':    ['pinnacle_A', 'bet365_A', 'williamhill_A'],
    'WHH':    ['williamhill_H', 'bet365_H'],
    'WHD':    ['williamhill_D', 'bet365_D'],
    'WHA':    ['williamhill_A', 'bet365_A'],
    'LBH':    ['ladbrokes_uk_H', 'bet365_H'],
    'LBD':    ['ladbrokes_uk_D', 'bet365_D'],
    'LBA':    ['ladbrokes_uk_A', 'bet365_A'],
    'GBH':    ['gamebookers_H', 'bet365_H'],
    'GBD':    ['gamebookers_D', 'bet365_D'],
    'GBA':    ['gamebookers_A', 'bet365_A'],
    'B365>2.5': ['bet365_>2.5', 'pinnacle_>2.5', 'betway_>2.5'],
    'B365<2.5': ['bet365_<2.5', 'pinnacle_<2.5', 'betway_<2.5'],
    'P>2.5':    ['pinnacle_>2.5', 'bet365_>2.5', 'betway_>2.5'],
    'P<2.5':    ['pinnacle_<2.5', 'bet365_<2.5', 'betway_<2.5'],
    # etc
}

for col in future_fixture_df.columns:
    if col in market_map and col in merged.columns:
        for fallback in market_map[col]:
            if fallback in merged.columns:
                merged[col] = merged[col].fillna(merged[fallback])

for col in future_fixture_df.columns:
    if col in merged.columns:
        
        if col.endswith('H') and not any(x in col for x in ('AH', 'SH', 'CH', 'HH')) and col not in ['HomeTeam']:
            poss = [c for c in merged.columns if c.endswith('_H') and c != col]
            mask_empty = merged[col].isna()
            for fallback in poss:
                merged.loc[mask_empty, col] = merged.loc[mask_empty, col].fillna(merged.loc[mask_empty, fallback])
                mask_empty = merged[col].isna()
       
        elif col.endswith('D') and not any(x in col for x in ('AD', 'AHD')) and col not in ['DayOfWeek']:
            poss = [c for c in merged.columns if c.endswith('_D') and c != col]
            mask_empty = merged[col].isna()
            for fallback in poss:
                merged.loc[mask_empty, col] = merged.loc[mask_empty, col].fillna(merged.loc[mask_empty, fallback])
                mask_empty = merged[col].isna()
       
        elif col.endswith('A') and not any(x in col for x in ('HA', 'AHA')) and col not in ['AwayTeam']:
            poss = [c for c in merged.columns if c.endswith('_A') and c != col]
            mask_empty = merged[col].isna()
            for fallback in poss:
                merged.loc[mask_empty, col] = merged.loc[mask_empty, col].fillna(merged.loc[mask_empty, fallback])
                mask_empty = merged[col].isna()
       
        elif '>2.5' in col:
            poss = [c for c in merged.columns if '>2.5' in c and c != col]
            mask_empty = merged[col].isna()
            for fallback in poss:
                merged.loc[mask_empty, col] = merged.loc[mask_empty, col].fillna(merged.loc[mask_empty, fallback])
                mask_empty = merged[col].isna()
       
        elif '<2.5' in col:
            poss = [c for c in merged.columns if '<2.5' in c and c != col]
            mask_empty = merged[col].isna()
            for fallback in poss:
                merged.loc[mask_empty, col] = merged.loc[mask_empty, col].fillna(merged.loc[mask_empty, fallback])
                mask_empty = merged[col].isna()

final_columns = list(future_fixture_df.columns)
final_df = merged[final_columns]

final_df.to_csv('updated_fixtures_with_odds.csv', index=False)
print("✅ Done! All FUTURE odds columns are now filled as much as possible (using best available odd for each market).")

odds_cols = [c for c in final_df.columns if c.upper() not in ("HOMETEAM", "AWAYTEAM", "DATE")]
print(final_df[['HomeTeam', 'AwayTeam', 'Date'] + odds_cols[:10]].head(10))

print("Requests remaining:", response.headers.get("x-requests-remaining"))
print("Requests used:", response.headers.get("x-requests-used"))
print("Requests cost for this call:", response.headers.get("x-requests-last"))

/var/folders/rj/ybzbgv510dd6rzmm3p9vf0000000gn/T/ipykernel_88004/4169307069.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fixture_df['date'] = pd.to_datetime(fixture_df['Date']).dt.tz_localize(None)
/var/folders/rj/ybzbgv510dd6rzmm3p9vf0000000gn/T/ipykernel_88004/4169307069.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fixture_df['home_team'] = fixture_df['HomeTeam'].str.strip().str.lower()
/var/folders/rj/ybzbgv510dd6rzmm3p9vf0000000gn/T/ipykernel_88004/4169307069.py:13: PerformanceWarning: DataFrame is highly fra

✅ Done! All FUTURE odds columns are now filled as much as possible (using best available odd for each market).
         HomeTeam     AwayTeam                 Date  Year  Month  DayOfWeek  \
0       Brentford       Fulham  2026-04-18 12:30:00  2026      4          5   
1           Leeds       Wolves  2026-04-18 15:00:00  2026      4          5   
2       Newcastle  Bournemouth  2026-04-18 15:00:00  2026      4          5   
3       Tottenham     Brighton  2026-04-18 17:30:00  2026      4          5   
4         Chelsea   Man United  2026-04-18 20:00:00  2026      4          5   
5     Aston Villa   Sunderland  2026-04-19 14:00:00  2026      4          6   
6         Everton    Liverpool  2026-04-19 14:00:00  2026      4          6   
7   Nott'm Forest      Burnley  2026-04-19 14:00:00  2026      4          6   
8        Man City      Arsenal  2026-04-19 16:30:00  2026      4          6   
9  Crystal Palace     West Ham  2026-04-20 20:00:00  2026      4          0   

   Div  Time  FTHG 